In [36]:
import chromadb
import re
from typing import List, Dict, Any
from dataclasses import dataclass

@dataclass
class ClassChunk:
    """Represents a chunk of class information"""
    class_name: str
    chunk_type: str  # 'overview', 'features', 'progression', 'proficiencies', 'equipment'
    content: str
    level_range: str = "All"
    metadata: Dict[str, Any] = None

class ClassChunkProcessor:
    def __init__(self):
        self.chunks = []

    def parse_classes_text(self, text: str) -> List[ClassChunk]:
        """Simple, robust parsing that won't get stuck in loops"""

        print("🔍 Parsing classes text...")
        self.chunks = []  # Reset chunks

        # Known D&D classes in order
        class_names = ['BARBARIAN', 'BARD', 'CLERIC', 'DRUID', 'FIGHTER',
                      'MONK', 'PALADIN', 'RANGER', 'ROGUE', 'SORCERER',
                      'WARLOCK', 'WIZARD']

        for i, class_name in enumerate(class_names):
            print(f"  Processing {class_name}...")

            # Find start of this class
            start_pattern = rf'\b{class_name}\b'
            start_match = re.search(start_pattern, text, re.IGNORECASE)

            if not start_match:
                print(f"    ❌ Could not find {class_name}")
                continue

            start_pos = start_match.start()

            # Find end of this class (start of next class or end of text)
            end_pos = len(text)
            for j in range(i + 1, len(class_names)):
                next_class = class_names[j]
                next_pattern = rf'\b{next_class}\b'
                next_match = re.search(next_pattern, text[start_pos + 100:])  # Skip first 100 chars
                if next_match:
                    end_pos = start_pos + 100 + next_match.start()
                    break

            # Extract this class's text
            class_text = text[start_pos:end_pos].strip()

            if len(class_text) > 200:  # Make sure we got meaningful content
                # Create chunks for this class
                self._create_simple_chunks(class_name, class_text)
                print(f"    ✅ Processed {class_name} ({len(class_text)} chars)")
            else:
                print(f"    ❌ {class_name} content too short ({len(class_text)} chars)")

        print(f"✅ Parsing complete: {len(self.chunks)} total chunks created")
        return self.chunks

    def _create_simple_chunks(self, class_name: str, class_text: str):
        """Create simple chunks from class text"""

        # 1. Full class overview (first 1500 chars as overview)
        overview_content = class_text[:1500]
        self.chunks.append(ClassChunk(
            class_name=class_name.title(),
            chunk_type='overview',
            content=overview_content,
            metadata={'section': 'description'}
        ))

        # 2. Look for proficiencies section
        prof_match = re.search(r'PROFICIENCIES(.*?)(?=EQUIPMENT|$)', class_text, re.DOTALL | re.IGNORECASE)
        if prof_match:
            prof_content = prof_match.group(0)[:800]  # Limit size
            self.chunks.append(ClassChunk(
                class_name=class_name.title(),
                chunk_type='proficiencies',
                content=prof_content,
                metadata={'section': 'proficiencies'}
            ))

        # 3. Look for equipment section
        equip_match = re.search(r'EQUIPMENT(.*?)(?=\n[A-Z]{3,}|$)', class_text, re.DOTALL | re.IGNORECASE)
        if equip_match:
            equip_content = equip_match.group(0)[:600]  # Limit size
            self.chunks.append(ClassChunk(
                class_name=class_name.title(),
                chunk_type='equipment',
                content=equip_content,
                metadata={'section': 'equipment'}
            ))

        # 4. Create feature chunks (split by common feature patterns)
        feature_sections = re.split(r'\n([A-Z][A-Z\s]{3,})\n', class_text[1500:])  # Skip overview part

        for i in range(1, len(feature_sections), 2):  # Every other element is a header
            if i + 1 < len(feature_sections):
                feature_name = feature_sections[i].strip()
                feature_content = feature_sections[i + 1].strip()

                if len(feature_content) > 50 and len(feature_name) < 50:  # Reasonable size
                    self.chunks.append(ClassChunk(
                        class_name=class_name.title(),
                        chunk_type='features',
                        content=f"{feature_name}\n{feature_content[:1000]}",  # Limit content
                        level_range=self._extract_level_from_content(feature_content),
                        metadata={
                            'feature_name': feature_name,
                            'section': 'class_features'
                        }
                    ))

    def _extract_level_from_content(self, content: str) -> str:
        """Extract level information from feature content"""

        level_patterns = [
            r'(\d+(?:st|nd|rd|th))\s+level',
            r'Starting at (\d+(?:st|nd|rd|th)) level',
            r'At (\d+(?:st|nd|rd|th)) level',
        ]

        for pattern in level_patterns:
            match = re.search(pattern, content, re.IGNORECASE)
            if match:
                return match.group(1)

        return "All"

# Initialize the processor
processor = ClassChunkProcessor()
print("✅ Replacement Class chunk processor ready!")
print("Usage: chunks = processor.parse_classes_text(classes_text)")

✅ Replacement Class chunk processor ready!
Usage: chunks = processor.parse_classes_text(classes_text)


In [37]:
import chromadb
from chromadb.config import Settings

def setup_classes_collection():
    """Setup ChromaDB collection for D&D classes"""

    # Initialize ChromaDB client
    client = chromadb.PersistentClient(
        path="./chromadb",
        settings=Settings(allow_reset=True)
    )

    # Create or get classes collection
    try:
        collection = client.get_collection("classes")
        print("📚 Found existing classes collection")
    except:
        collection = client.create_collection(
            name="classes",
            metadata={"description": "D&D 5e class information and features"}
        )
        print("📚 Created new classes collection")

    return client, collection

def add_classes_to_chromadb(chunks, collection):
    """Add class chunks to ChromaDB collection"""

    documents = []
    metadatas = []
    ids = []

    for i, chunk in enumerate(chunks):
        # Create document text
        doc_text = f"Class: {chunk.class_name}\n"
        doc_text += f"Type: {chunk.chunk_type}\n"
        doc_text += f"Level Range: {chunk.level_range}\n\n"
        doc_text += chunk.content

        documents.append(doc_text)

        # Create metadata
        metadata = {
            'class_name': chunk.class_name,
            'chunk_type': chunk.chunk_type,
            'level_range': chunk.level_range,
            'source': 'players_handbook_classes'
        }

        # Add any additional metadata
        if chunk.metadata:
            metadata.update(chunk.metadata)

        metadatas.append(metadata)

        # Create unique ID
        ids.append(f"class_{chunk.class_name.lower()}_{chunk.chunk_type}_{i}")

    # Add to collection
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )

    print(f"✅ Added {len(chunks)} class chunks to ChromaDB")
    return len(chunks)

def query_classes_collection(collection, query_text, class_filter=None, chunk_type_filter=None, n_results=5):
    """Query the classes collection"""

    # Build where clause for filtering
    where_clause = {}
    if class_filter:
        where_clause['class_name'] = class_filter
    if chunk_type_filter:
        where_clause['chunk_type'] = chunk_type_filter

    # Perform query
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        where=where_clause if where_clause else None
    )

    return results

print("✅ ChromaDB classes setup functions ready!")
print("Usage:")
print("  client, collection = setup_classes_collection()")
print("  add_classes_to_chromadb(chunks, collection)")
print("  results = query_classes_collection(collection, 'rage barbarian')")

✅ ChromaDB classes setup functions ready!
Usage:
  client, collection = setup_classes_collection()
  add_classes_to_chromadb(chunks, collection)
  results = query_classes_collection(collection, 'rage barbarian')


In [38]:
import pdfplumber
import os

def extract_classes_simple(pdf_path: str, start_page: int = 46, end_page: int = 121) -> str:
    """Simple extraction from page 4 to 121"""

    if not os.path.exists(pdf_path):
        print(f"❌ Error: PDF file not found at {pdf_path}")
        return None

    print(f"📖 Extracting pages {start_page} to {end_page} from PDF...")

    classes_text = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num in range(start_page, end_page + 1):
                if page_num > len(pdf.pages):
                    break

                if page_num % 10 == 0:
                    print(f"   Page {page_num}...")

                page = pdf.pages[page_num - 1]  # 0-indexed
                page_text = page.extract_text()

                if page_text:
                    classes_text += page_text + "\n"

            print(f"✅ Extracted {len(classes_text)} characters from pages {start_page}-{end_page}")
            return classes_text

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def save_extracted_classes(text: str, filename: str = "extracted_classes.txt"):
    """Save extracted classes text to file"""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(text)
    print(f"💾 Classes text saved to {filename}")

# Test the simple extractor
print("✅ Simple PDF extractor ready!")
print("Usage: classes_text = extract_classes_simple('your_pdf.pdf')")

✅ Simple PDF extractor ready!
Usage: classes_text = extract_classes_simple('your_pdf.pdf')


In [39]:
def process_classes_pdf_to_chromadb(pdf_path: str):
    """Complete pipeline: PDF -> chunks -> ChromaDB"""

    print("🚀 Starting classes processing pipeline...")

    # Step 1: Extract text from PDF
    print("\n📖 Step 1: Extracting text from PDF...")
    classes_text = extract_classes_simple(pdf_path)

    if not classes_text:
        print("❌ Failed to extract classes text from PDF")
        return None

    # Optional: Save extracted text as backup
    save_extracted_classes(classes_text)

    # Step 2: Process into chunks
    print("\n📋 Step 2: Processing into chunks...")
    processor = ClassChunkProcessor()
    chunks = processor.parse_classes_text(classes_text)

    if not chunks:
        print("❌ No chunks created from classes text")
        return None

    print(f"✅ Created {len(chunks)} class chunks")

    # Show sample of what we created
    print("\n🔍 Sample chunks by type:")
    chunk_types = {}
    for chunk in chunks:
        if chunk.chunk_type not in chunk_types:
            chunk_types[chunk.chunk_type] = []
        chunk_types[chunk.chunk_type].append(chunk)

    for chunk_type, type_chunks in chunk_types.items():
        print(f"  {chunk_type}: {len(type_chunks)} chunks")
        if type_chunks:
            sample = type_chunks[0]
            print(f"    Sample: {sample.class_name} - {sample.content[:60]}...")

    # Step 3: Setup ChromaDB and add chunks
    print("\n💾 Step 3: Adding to ChromaDB...")
    client, collection = setup_classes_collection()
    total_added = add_classes_to_chromadb(chunks, collection)

    # Step 4: Test the collection
    print(f"\n🧪 Step 4: Testing collection...")
    print(f"Total documents in collection: {collection.count()}")

    # Test with a few queries
    test_queries = [
        "barbarian rage damage bonus",
        "cleric spellcasting ability",
        "fighter extra attack",
        "wizard spell preparation"
    ]

    for query in test_queries[:2]:  # Just test 2 to keep output manageable
        print(f"\n🔍 Test query: '{query}'")
        results = query_classes_collection(collection, query, n_results=1)

        if results['documents'][0]:
            doc = results['documents'][0][0]
            metadata = results['metadatas'][0][0]
            distance = results['distances'][0][0]

            print(f"  Best match: {metadata['class_name']} - {metadata['chunk_type']}")
            print(f"  Distance: {distance:.3f}")
            print(f"  Content: {doc[:150]}...")

    print(f"\n✅ Pipeline complete! Added {total_added} class chunks to ChromaDB")
    return client, collection, chunks

print("✅ Complete classes pipeline ready!")
print("Usage: client, collection, chunks = process_classes_pdf_to_chromadb('path/to/players_handbook.pdf')")

✅ Complete classes pipeline ready!
Usage: client, collection, chunks = process_classes_pdf_to_chromadb('path/to/players_handbook.pdf')


In [40]:
# Run the complete pipeline
#client, collection, chunks = process_classes_pdf_to_chromadb('Dungeons  Dragons 5e Players Handbook (Wizards RPG Team Wyatt James, Schwalb Robert J etc.) (Z-Library).pdf')

In [41]:
import pdfplumber

# Very simple approach - just extract the pages we need
pdf_path = 'Dungeons  Dragons 5e Players Handbook (Wizards RPG Team Wyatt James, Schwalb Robert J etc.) (Z-Library).pdf'

print("📖 Opening PDF...")
with pdfplumber.open(pdf_path) as pdf:
    classes_text = ""

    for page_num in range(46, 122):  # Pages 46-121 (range goes to 122)
        print(f"Page {page_num}")

        page = pdf.pages[page_num - 1]  # Convert to 0-indexed
        page_text = page.extract_text()

        if page_text:
            classes_text += page_text + "\n"

print(f"✅ Extracted {len(classes_text)} characters")

# Save it
#with open("classes_extracted.txt", "w", encoding="utf-8") as f:
 #   f.write(classes_text)

print("💾 Saved to classes_extracted.txt")

📖 Opening PDF...
Page 46
Page 47
Page 48
Page 49
Page 50
Page 51
Page 52
Page 53
Page 54
Page 55
Page 56
Page 57
Page 58
Page 59
Page 60
Page 61
Page 62
Page 63
Page 64
Page 65
Page 66
Page 67
Page 68
Page 69
Page 70
Page 71
Page 72
Page 73
Page 74
Page 75
Page 76
Page 77
Page 78
Page 79
Page 80
Page 81
Page 82
Page 83
Page 84
Page 85
Page 86
Page 87
Page 88
Page 89
Page 90
Page 91
Page 92
Page 93
Page 94
Page 95
Page 96
Page 97
Page 98
Page 99
Page 100
Page 101
Page 102
Page 103
Page 104
Page 105
Page 106
Page 107
Page 108
Page 109
Page 110
Page 111
Page 112
Page 113
Page 114
Page 115
Page 116
Page 117
Page 118
Page 119
Page 120
Page 121
✅ Extracted 285334 characters
💾 Saved to classes_extracted.txt


In [42]:
# Load the extracted classes text
with open('classes_extracted.txt', 'r', encoding='utf-8') as f:
    classes_text = f.read()

print(f"📖 Loaded {len(classes_text)} characters from classes_extracted.txt")

# Process into chunks using our existing processor
processor = ClassChunkProcessor()
chunks = processor.parse_classes_text(classes_text)

print(f"📋 Created {len(chunks)} class chunks")

# Show what we got
chunk_types = {}
for chunk in chunks:
    if chunk.chunk_type not in chunk_types:
        chunk_types[chunk.chunk_type] = []
    chunk_types[chunk.chunk_type].append(chunk)

print("\n🔍 Chunks by type:")
for chunk_type, type_chunks in chunk_types.items():
    print(f"  {chunk_type}: {len(type_chunks)} chunks")

    # Show sample classes for this type
    classes_in_type = set([c.class_name for c in type_chunks])
    print(f"    Classes: {', '.join(sorted(classes_in_type))}")

print(f"\n📊 Total chunks: {len(chunks)}")

📖 Loaded 283257 characters from classes_extracted.txt
🔍 Parsing classes text...
  Processing BARBARIAN...
    ✅ Processed BARBARIAN (17406 chars)
  Processing BARD...
    ✅ Processed BARD (33858 chars)
  Processing CLERIC...
    ✅ Processed CLERIC (65137 chars)
  Processing DRUID...
    ✅ Processed DRUID (86733 chars)
  Processing FIGHTER...
    ✅ Processed FIGHTER (113124 chars)
  Processing MONK...
    ✅ Processed MONK (133335 chars)
  Processing PALADIN...
    ✅ Processed PALADIN (160729 chars)
  Processing RANGER...
    ✅ Processed RANGER (179009 chars)
  Processing ROGUE...
    ✅ Processed ROGUE (197256 chars)
  Processing SORCERER...
    ✅ Processed SORCERER (219464 chars)
  Processing WARLOCK...
    ✅ Processed WARLOCK (247657 chars)
  Processing WIZARD...
    ✅ Processed WIZARD (177813 chars)
✅ Parsing complete: 1863 total chunks created
📋 Created 1863 class chunks

🔍 Chunks by type:
  overview: 12 chunks
    Classes: Barbarian, Bard, Cleric, Druid, Fighter, Monk, Paladin, Rang

In [43]:
# Add the chunks to ChromaDB
client, collection = setup_classes_collection()
total_added = add_classes_to_chromadb(chunks, collection)

print(f"\n📊 Collection stats:")
print(f"Total documents: {collection.count()}")

# Test with D&D-specific queries
test_queries = [
    "barbarian rage damage bonus",
    "wizard spell preparation",
    "cleric healing abilities",
    "rogue sneak attack",
    "fighter extra attack",
    "paladin divine smite",
    "monk ki points",
    "bard bardic inspiration"
]

print(f"\n🧪 Testing {len(test_queries)} queries...")

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = query_classes_collection(collection, query, n_results=3)

    for i, doc in enumerate(results['documents'][0]):
        metadata = results['metadatas'][0][i]
        distance = results['distances'][0][i]

        print(f"  {i+1}. {metadata['class_name']} - {metadata['chunk_type']}")
        print(f"     Distance: {distance:.3f}")
        print(f"     Preview: {doc[:120]}...")
        print()

📚 Created new classes collection
✅ Added 1863 class chunks to ChromaDB

📊 Collection stats:
Total documents: 1863

🧪 Testing 8 queries...

🔍 Query: 'barbarian rage damage bonus'
  1. Barbarian - features
     Distance: 0.829
     Preview: Class: Barbarian
Type: features
Level Range: All

DANGER SENSE
When you make amelee weapon attack using
Strcngth, you ga...

  2. Barbarian - features
     Distance: 0.844
     Preview: Class: Barbarian
Type: features
Level Range: All

PRIMAL PATH
knocked unconscious or ifyour turn ends and you
haven't at...

  3. Barbarian - features
     Distance: 0.870
     Preview: Class: Barbarian
Type: features
Level Range: All

RAGE
for defense toattack with fierce desperation. When
In battle, you...


🔍 Query: 'wizard spell preparation'
  1. Wizard - features
     Distance: 0.880
     Preview: Class: Wizard
Type: features
Level Range: All

PREPARING ANO CASTING SPELLS
(a)aeomponent poueh or (b)an areane foeus
Th...

  2. Wizard - features
     Distance: 0.904
